# NLP4 - Topic Modeling: NMF and LDA


In this session we will explore two techniques for discovering themes in text: Non-negative Matrix Factorization (NMF) and Latent Dirichlet Allocation (LDA). We begin with preprocessing, build an NMF model using TF-IDF features, and then use token counts to train and interpret an LDA model.

NMF represents a non-negative document-term matrix as document-topic and topic-term weights. LDA takes a probabilistic approach, representing documents as mixtures of topics and topics as distributions over words. The lab walks through both methods with discussion, step-by-step code, and interpretation.

---

## Install Libraries

In [1]:
#first lets install datasets library
# !pip install datasets
# !python -m spacy download pt_core_news_lg

## Imports

In [3]:
from itertools import count

#dataset library
from datasets import load_dataset

#load dataset
dataset = load_dataset("tclopess/sinopsys_movies_portuguese")
#convert it to pandas and slice the first 3000 data points
df_sinop = dataset['train'].to_pandas()[:3000]


#NLP tool box nltk
import nltk
from nltk.corpus import stopwords
#getting stop words
# nltk.download('stopwords')
stop = list(set(stopwords.words('portuguese')))
print(stop)

#string library
import string
#get list of punctuations
pontuacoes = string.punctuation
print(pontuacoes)

#NLP toolbox spacy
import spacy
#load portuguese module large
nlp = spacy.load("pt_core_news_lg")

#other python support libraries and methods
import itertools
from collections import Counter
from collections import defaultdict
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

#dataframes library
import pandas as pd

#LDA library
import gensim
import gensim.corpora as corpora
from gensim.models.ldamodel import LdaModel

/Users/zuilpirola/PyCharmMiscProject/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


['aquela', 'ou', 'tenho', 'éramos', 'formos', 'sejam', 'estas', 'haja', 'estivéramos', 'da', 'houveríamos', 'os', 'nos', 'fosse', 'seja', 'das', 'com', 'ele', 'houvessem', 'houvéssemos', 'ser', 'serei', 'estava', 'também', 'quando', 'numa', 'pelos', 'tua', 'sou', 'tivéramos', 'seus', 'tenhamos', 'seremos', 'tiver', 'forem', 'teríamos', 'você', 'entre', 'nem', 'tivera', 'tiverem', 'já', 'tiveram', 'tenha', 'for', 'tém', 'estivesse', 'sem', 'dela', 'esteja', 'houverem', 'pelo', 'estivemos', 'ao', 'fossem', 'uma', 'aos', 'muito', 'esses', 'tive', 'depois', 'e', 'elas', 'hão', 'esta', 'estivessem', 'quem', 'lhe', 'seria', 'do', 'são', 'sua', 'tuas', 'estejamos', 'para', 'pelas', 'deles', 'estar', 'em', 'nossas', 'estiverem', 'fôramos', 'qual', 'às', 'minhas', 'houvesse', 'será', 'fomos', 'há', 'meu', 'que', 'terei', 'estiveram', 'tínhamos', 'nossa', 'fora', 'estivéssemos', 'estamos', 'houvermos', 'a', 'esse', 'no', 'nosso', 'estejam', 'tivéssemos', 'tivemos', 'estão', 'fôssemos', 'seu', 'h

In [4]:
df_sinop.head()

,titulo,sinopse,generos,is_valid
0,We Were Soldiers,A história da primeira grande batalha da fase ...,"['Ação', 'História', 'Guerra']",False
1,4Got10,"Um negócio de drogas dá errado, deixando corpo...","['Ação', 'Crime', 'Thriller']",False
2,Pontypool,Quando o disc jockey Grant Mazzy se reporta à ...,"['Horror', 'Mistério', 'Ficção Científica']",False
3,Ticker,Depois que o parceiro de um detetive de São Fr...,"['Ação', 'Crime', 'Thriller']",False
4,Real Genius,Um adolescente prodígio tenso entra em uma fac...,"['Comédia', 'Romance', 'Ficção Científica']",True


### Practicing


1 - Using the concepts from the previous class, create a function that takes a string as a parameter and returns a list of pre-processed tokens. The tokens should be lowercase, lemmas, and must not be punctuation or stopwords.

In [7]:
#funcao de preprocessing com lemmas
def preprocessing(text):
  return [x.lemma_.lower() for x in nlp(text) if x.text.lower() not in stop and x.text not in pontuacoes]

#nessa funcao temos lemma , lower, stopwords e pontuacoes


2 - Using the function you created in the previous exercise, preprocess all the synopsis texts contained in the dataframe.

In [8]:
#exemplo das sinopses
df_sinop.head()

,titulo,sinopse,generos,is_valid
0,We Were Soldiers,A história da primeira grande batalha da fase ...,"['Ação', 'História', 'Guerra']",False
1,4Got10,"Um negócio de drogas dá errado, deixando corpo...","['Ação', 'Crime', 'Thriller']",False
2,Pontypool,Quando o disc jockey Grant Mazzy se reporta à ...,"['Horror', 'Mistério', 'Ficção Científica']",False
3,Ticker,Depois que o parceiro de um detetive de São Fr...,"['Ação', 'Crime', 'Thriller']",False
4,Real Genius,Um adolescente prodígio tenso entra em uma fac...,"['Comédia', 'Romance', 'Ficção Científica']",True


In [9]:
#aplicar a funcao de preprocessing em todas as sinopses
preprocessed_docs = [preprocessing(x) for x in df_sinop['sinopse'].to_list()]

In [10]:
#exemplo de uma sinopse preprocessada
preprocessed_docs[0]

['história',
 'primeiro',
 'grande',
 'batalha',
 'fase',
 'americano',
 'guerra',
 'vietnã',
 'soldado',
 'ambos',
 'lado',
 'travar']

In [11]:
#outra forma de aplicar a funcao de preprocessing em todas as sinopses
#create a new column with preprocessed texts
df_sinop['tokens'] = df_sinop['sinopse'].apply(preprocessing)


## Non-negative Matrix Factorization (NMF)

NMF is a matrix factorization method. Given a non-negative document-term matrix **X**, it approximates the matrix as **W × H**. Here, rows of **W** describe documents in terms of topics, while rows of **H** describe topics in terms of words. With text, TF-IDF is a common input: each document is represented by the importance of its terms.

Unlike LDA, NMF is not a probabilistic generative model. Its topic weights are non-negative and useful for ranking, but they are not automatically probabilities that sum to one. Topic labels still require human interpretation.

### 1 — Prepare a TF-IDF matrix from the tokens

The corpus has already been lowercased, lemmatized, and filtered for stopwords above. Join each token list into a string, then let `TfidfVectorizer` build a document-term matrix. `min_df=5` removes terms found in fewer than five documents, reducing noise and vocabulary size. What are the rows and columns of this matrix?

In [12]:
from sklearn.feature_extraction.text import TfidfVectorizer

#reconstruindo os textos a partir dos tokens
texts_for_nmf = [" ".join(tokens) for tokens in preprocessed_docs]

#criando a matriz tfidf
tfidf_vectorizer = TfidfVectorizer(min_df=5, max_df=0.90)
X_tfidf = tfidf_vectorizer.fit_transform(texts_for_nmf)
feature_names = tfidf_vectorizer.get_feature_names_out()

print("Matrix shape (documents, terms):", X_tfidf.shape)
print("First document has", X_tfidf[0].nnz, "non-zero term weights")

Matrix shape (documents, terms): (3000, 2483)
First document has 11 non-zero term weights


**Solution / interpretation.** Each row is one synopsis; each column is a vocabulary term. A non-zero cell is that term’s TF-IDF weight in that synopsis. The matrix is sparse because any one document contains only a small part of the full vocabulary.

### 2 — Choose the number of topics and fit NMF

Set `n_components` to the number of topics to discover. There is no universally correct value: compare several values, inspect whether topics are coherent and distinct, and use domain knowledge. We start with 10 to make the result comparable to the LDA example. `random_state` makes initialization reproducible; `max_iter` sets the iteration limit.

In [13]:
#definindo o K com base na quantidade de generos de filmes que temos no dataset
#converter a listas em formato string para lista
df_sinop['generos'] = df_sinop['generos'].apply(lambda x: eval(x) if isinstance(x, str) else x)
#fazendo um flatten na lista de generos
all_genres = list(itertools.chain.from_iterable(df_sinop['generos'].to_list()))
#contando a quantidade de generos unicos
unique_genres = set(all_genres)
print("Unique genres:", unique_genres)
len(unique_genres)


Unique genres: {'Guerra', 'Música', 'Crime', 'Filme de TV', 'Documentário', 'História', 'Thriller', 'Ficção Científica', 'Esporte', 'Fantasia', 'Filme Noir', 'Drama', 'Mistério', 'Horror', 'Animação', 'Romance', 'Aventura', 'Ocidental', 'Estrangeiro', 'Comédia', 'Família', 'Biografia', 'Ação'}


23

In [14]:
from sklearn.decomposition import NMF

n_topics = 23
nmf_model = NMF(
    n_components=n_topics,
    init="nndsvda",
    random_state=100,
    max_iter=400
)
W = nmf_model.fit_transform(X_tfidf)
H = nmf_model.components_
print("Document-topic matrix W:", W.shape)
print("Topic-term matrix H:", H.shape)
print("Reconstruction error:", nmf_model.reconstruction_err_)

Document-topic matrix W: (3000, 23)
Topic-term matrix H: (23, 2483)
Reconstruction error: 52.52579093694941


**Solution / interpretation.** `W[d, k]` is the weight of topic `k` in document `d`; `H[k, t]` is the weight of term `t` in topic `k`. Both are non-negative. The reconstruction error summarizes how closely `W @ H` approximates the original TF-IDF matrix; it is useful for comparing runs with the same input, but lower error alone does not guarantee more interpretable topics.

### 3 — Inspect the most important words in each topic

For each row in **H**, sort term weights from largest to smallest. The top words help us assign a human-readable label to the topic. Are the words related, and do two topics appear to describe the same theme?

In [15]:
def show_nmf_topics(model, feature_names, n_top_words=12):
    for topic_idx, topic_weights in enumerate(model.components_):
        top_indices = topic_weights.argsort()[-n_top_words:][::-1]
        words = [feature_names[i] for i in top_indices]
        print(f"Topic {topic_idx}: {' | '.join(words)}")

show_nmf_topics(nmf_model, feature_names)

Topic 0: fazer | amizade | ex | tudo | poder | coisa | viagem | grande | decidir | qualquer | presidiário | filha
Topic 1: nenhum | geral | visão | estranho | conflito | envolvir | caso | três | arquivo | 14 | motivo | lembrar
Topic 2: guerra | durante | mundial | segunda | americano | soldado | exército | missão | britânico | oficial | eua | civil
Topic 3: história | contar | amor | real | baseado | verdadeiro | the | americano | cujo | final | lenda | animar
Topic 4: policial | assassino | assassinato | agente | criminoso | crime | detetive | investigar | polícia | matar | ex | los
Topic 5: dois | dia | irmão | irmã | outro | diferente | gangue | seguir | casamento | amoroso | bandido | rival
Topic 6: vida | amor | sobre | mudar | viver | levar | casal | estilo | pessoa | virar | cabeça | baixo
Topic 7: homem | matar | outro | acidente | deixar | perder | mudar | carro | personalidade | sequestrar | resgatar | envolver
Topic 8: cidade | nova | york | pequeno | grande | chegar | natal

In [16]:
#candidatos a topico romance
#topico 6, 12 20

**Solution / interpretation.** Read each word list as a set of terms that jointly characterize a topic, not as a guaranteed label. Give a topic a short label only after reading representative documents. Repeated or vague terms can indicate that the topic count, preprocessing, or term filtering needs adjustment.

### 4 — Rank topics for each document

Use the corresponding row of **W** to see which topics are strongest in a synopsis. These are weights, not probabilities. We can normalize a row to sum to one for a relative comparison within that document; this normalization is a convenient display choice, not a probabilistic guarantee from NMF.

In [17]:
#id do document to analyze
doc_id = 0
#obtemos os pesos do documento
doc_weights = W[doc_id]
#calculamos os pesos relativos
relative_weights = doc_weights / doc_weights.sum() if doc_weights.sum() else doc_weights
#ordenamos os topicos pelo peso relativo
ranked_topics = relative_weights.argsort()[::-1]

print(df_sinop.iloc[doc_id]["sinopse"][:500], "...\n")
for topic_idx in ranked_topics[:3]:
    print(f"Topic {topic_idx}: relative weight={relative_weights[topic_idx]:.3f}")

A história da primeira grande batalha da fase americana da Guerra do Vietnã e os soldados de ambos os lados que a travaram. ...

Topic 2: relative weight=0.502
Topic 3: relative weight=0.431
Topic 22: relative weight=0.021


### 5 — Find representative documents for a topic

A topic’s top words give one view; documents with the largest topic weights give another. Inspecting both helps validate our interpretation and spot topics that mix unrelated themes.

In [18]:
topic_id = 0
top_doc_ids = W[:, topic_id].argsort()[-5:][::-1]
for doc_id in top_doc_ids:
    print(f"\nDocument {doc_id} | topic weight={W[doc_id, topic_id]:.3f}")
    print(df_sinop.iloc[doc_id]["sinopse"][:350], "...")


Document 2072 | topic weight=0.066
Um ex-presidiário faz amizade com um garoto de um lar problemático. ...

Document 965 | topic weight=0.062
Um órfão faz sacrifícios pela família que o acolheu, mas depois enfrenta humilhação e falta de moradia. ...

Document 542 | topic weight=0.061
A vida de um cara é revirada por um e-mail, que inclui os nomes de todos com quem ele fez sexo e com quem fará sexo. Sua situação piora quando ele encontra uma femme fatale (Ryder) que tem como alvo homens culpados de crimes sexuais. ...

Document 2595 | topic weight=0.057
Conheça Patch Adams, um médico que não se parece, age ou pensa como qualquer outro médico que você já conheceu. Para Patch, o humor é o melhor remédio, e ele está disposto a fazer qualquer coisa para fazer seus pacientes rirem - mesmo que isso signifique arriscar sua própria carreira. ...

Document 633 | topic weight=0.054
Nos últimos 40 anos, a Guerra às Drogas foi responsável por 45 milhões de prisões, fez da América o maior carcereir

**Discussion / solution.** Compare these synopses with the topic’s top words. If they share a recognizable subject, the interpretation is supported. If not, the topic may be broad, driven by common vocabulary, or one of several themes in the same documents. Topic modeling is exploratory: inspect examples before naming or using topics.

In [19]:
#analise do topico 6
topic_id = 6
top_doc_ids = W[:, topic_id].argsort()[-5:][::-1]
for doc_id in top_doc_ids:
    print(f"\nDocument {doc_id} | topic weight={W[doc_id, topic_id]:.3f}")
    print(df_sinop.iloc[doc_id]["sinopse"][:350], "...")


Document 45 | topic weight=0.171
O filme conta a história de um jovem que leva um estilo de vida promíscuo até que vários reveses de vida o fazem repensar seus propósitos e objetivos na vida. ...

Document 120 | topic weight=0.146
Barbara Thorson luta pela vida escapando para uma vida de fantasia de magia e monstros. ...

Document 897 | topic weight=0.141
Uma programadora de computador se depara com uma conspiração, colocando sua vida e a vida das pessoas ao seu redor em grande perigo. ...

Document 618 | topic weight=0.128
São apresentadas as vidas sexuais simultâneas dos melhores amigos Jonathan e Sandy, aquelas vidas que são afetadas pelos costumes sexuais da época e seu próprio temperamento, especialmente em relação às respectivas mulheres que acabam em suas vidas. ...

Document 475 | topic weight=0.126
A vida do boxeador Jake LaMotta, cuja violência e temperamento que o levaram ao topo no ringue destruíram sua vida fora dele. ...


In [20]:
#conferir para os topicos 6, 12 e 20 se o genero romance aparece na sinopse
for topic_id in [6, 12, 20]:
    top_doc_ids = W[:, topic_id].argsort()[-40:][::-1]
    count = 0
    for doc_id in top_doc_ids:
        # print(f"\nDocument {doc_id} | topic weight={W[doc_id, topic_id]:.3f}")
        if "Romance" in df_sinop.iloc[doc_id]["generos"]:
            count += 1

    print(f"Topic {topic_id} has {count} romance synopses in top 40 documents.")

Topic 6 has 9 romance synopses in top 40 documents.
Topic 12 has 11 romance synopses in top 40 documents.
Topic 20 has 16 romance synopses in top 40 documents.


## Topic Modeling with Latent Dirichlet Allocation (LDA)

NMF showed how a non-negative matrix can be represented through document-topic and topic-term weights. LDA approaches topic modeling probabilistically: it represents each document as a mixture of topics, and each topic as a distribution over words. We will build the inputs, train a model, inspect both distributions, discuss the results, and finish with an interactive pyLDAvis visualization.

LDA uses token counts rather than TF-IDF weights. We will reuse the same preprocessed tokens so the comparison with NMF is meaningful.

### 1 — Think about the LDA model

Before coding, discuss the central idea of the paper that introduced Latent Dirichlet Allocation: a document can contain several topics, and each topic can generate words with different probabilities. In your own words, describe how this differs from assigning each document to exactly one topic.

**Solution / discussion.** LDA treats each document as a mixture of topics. A synopsis might, for example, combine adventure and romance. Each topic is itself a probability distribution over words, so a word can be likely under more than one topic. The model infers these hidden distributions from word counts across the corpus.

### 2 — Build a word dictionary

Gensim needs a mapping between token strings and integer IDs. `corpora.Dictionary` builds this vocabulary from the preprocessed documents. This uses the same preprocessing as the TF-IDF and NMF sections: lowercase lemmas with stopwords and punctuation removed. Inspect a few ID-to-word mappings and the vocabulary size.

In [21]:
#create a dictionary from the preprocessed documents
id2word = corpora.Dictionary(preprocessed_docs)

print("Vocabulary size:", len(id2word))
print("Token with ID 0:", id2word[0])
print("First 10 ID-token pairs:", list(id2word.items())[:10])

Vocabulary size: 12603
Token with ID 0: ambos
First 10 ID-token pairs: [(0, 'ambos'), (1, 'americano'), (2, 'batalha'), (3, 'fase'), (4, 'grande'), (5, 'guerra'), (6, 'história'), (7, 'lado'), (8, 'primeiro'), (9, 'soldado')]


**Solution / interpretation.** The dictionary assigns each distinct token an integer ID. The IDs are bookkeeping keys; their numeric order has no semantic meaning. The dictionary allows Gensim to store documents compactly as pairs of token ID and count.

### 3 — Convert documents into bag-of-words counts

LDA models word occurrences, so convert each tokenized synopsis to a bag-of-words representation with `doc2bow`. Each document becomes a list of `(token_id, count)` pairs. Word order is not retained. Inspect the first document and map its IDs back to words.

In [22]:
corpus = [id2word.doc2bow(tokens) for tokens in preprocessed_docs]
print("Number of documents:", len(corpus))
print("First document as (token_id, count):", corpus[0][:10])
print("First document as (word, count):", [(id2word[token_id], count) for token_id, count in corpus[0][:10]])

Number of documents: 3000
First document as (token_id, count): [(0, 1), (1, 1), (2, 1), (3, 1), (4, 1), (5, 1), (6, 1), (7, 1), (8, 1), (9, 1)]
First document as (word, count): [('ambos', 1), ('americano', 1), ('batalha', 1), ('fase', 1), ('grande', 1), ('guerra', 1), ('história', 1), ('lado', 1), ('primeiro', 1), ('soldado', 1)]


**Solution / interpretation.** A pair such as `(12, 3)` means dictionary token 12 appears three times in that document. This representation differs from the NMF input: it contains raw token counts rather than TF-IDF weights.

### 4 — Choose the number of topics

`num_topics` is a modeling choice. Too few topics can merge distinct themes; too many can split a coherent theme into fragments. There is no universally correct number. In practice, compare several values, inspect topic coherence and redundancy, read representative documents, and use domain knowledge. For this worked example, we will fit 23 topics to match the NMF setting.

In [23]:
num_topics = 23
print("Topics to fit:", num_topics)

Topics to fit: 23


### 5 — Train the LDA model

`LdaModel` learns topic-word and document-topic distributions from the corpus. `passes` controls how many full passes are made over the corpus during training. `random_state` fixes the random seed for reproducibility. We use the same 23-topic setting as the NMF example, while remembering the models represent topic weights differently.

In [24]:
lda_model = LdaModel(
    corpus=corpus,
    id2word=id2word,
    num_topics=num_topics,
    passes=20,
    random_state=50
)
print("LDA model trained with", lda_model.num_topics, "topics")

LDA model trained with 23 topics


**Solution / interpretation.** `passes=20` means the training algorithm revisits the full corpus 20 times. More passes can improve learning but take longer; additional passes do not guarantee more interpretable topics. `random_state=100` makes the model initialization repeatable under the same software and data conditions.

### 6 — Inspect topic-word distributions (topic loadings)

Each LDA topic is a probability distribution over vocabulary terms. Print the 15 highest-probability words for each topic. The listed numbers are probabilities for words within each topic; they are not document-topic scores.

In [25]:
topic_terms = lda_model.print_topics(num_topics=num_topics, num_words=15)
for topic_id, terms in topic_terms:
    print(f"Topic {topic_id}: {terms}")

Topic 0: 0.021*"vida" + 0.013*"jovem" + 0.012*"história" + 0.008*"amigo" + 0.007*"amor" + 0.007*"enquanto" + 0.006*"ano" + 0.006*"descobrir" + 0.006*"grande" + 0.006*"mulher" + 0.006*"começar" + 0.005*"parte" + 0.005*"sobre" + 0.005*"romance" + 0.005*"dois"
Topic 1: 0.020*"história" + 0.016*"vida" + 0.008*"ano" + 0.008*"contar" + 0.007*"mulher" + 0.006*"começar" + 0.005*"enquanto" + 0.005*"escândalo" + 0.005*"viver" + 0.005*"sobre" + 0.005*"real" + 0.005*"ilha" + 0.005*"incluir" + 0.005*"nome" + 0.004*"bebê"
Topic 2: 0.011*"detetive" + 0.009*"navio" + 0.009*"encontrar" + 0.007*"assassino" + 0.006*"americano" + 0.006*"liderar" + 0.006*"nova" + 0.006*"preso" + 0.006*"zumbi" + 0.006*"york" + 0.005*"tripulação" + 0.005*"missão" + 0.005*"10" + 0.005*"aterrorizar" + 0.005*"fazer"
Topic 3: 0.010*"descobrir" + 0.010*"agente" + 0.009*"..." + 0.008*"agora" + 0.007*"killer" + 0.007*"secreto" + 0.007*"missão" + 0.007*"serial" + 0.007*"sobre" + 0.006*"enviar" + 0.006*"dar" + 0.006*"grande" + 0.006*

**Solution / interpretation.** The words with the largest probabilities help us describe a topic. A topic might be labeled “family” if its high-probability words and representative documents consistently support that interpretation. Labels are our interpretation, not names discovered automatically by the model.

In [26]:
#romance topics
#8, 0 ,13

In [35]:
#check if Romance appears in the top words for topics 8, 0, and 13

for topic_id in [1, 9, 13, 18, 21]:
    count =0
    #get the top documents for the topic
    top_doc_ids = sorted(range(len(corpus)), key=lambda d: lda_model.get_document_topics(corpus[d], minimum_probability=0)[topic_id][1], reverse=True)[:40]
    print(f"\nTopic {topic_id} top documents:")
    for doc_id in top_doc_ids:
        if "Romance" in df_sinop.iloc[doc_id]["generos"]:
            count += 1

    print(f"Number of documents with Romance genre in top 40 for topic {topic_id}: {count}")


Topic 1 top documents:
Number of documents with Romance genre in top 40 for topic 1: 4

Topic 9 top documents:
Number of documents with Romance genre in top 40 for topic 9: 6

Topic 13 top documents:
Number of documents with Romance genre in top 40 for topic 13: 6

Topic 18 top documents:
Number of documents with Romance genre in top 40 for topic 18: 7

Topic 21 top documents:
Number of documents with Romance genre in top 40 for topic 21: 5


### 7 — Inspect topic mixtures for documents (document scores)

For each document, LDA estimates a distribution over topics. `get_document_topics` returns `(topic_id, probability)` pairs. By default, very small probabilities may be omitted; set `minimum_probability=0` when you want all topics included. First inspect one synopsis and its topic distribution.

In [28]:
doc_id = 0
print(df_sinop.iloc[doc_id]["sinopse"][:500], "...\n")
doc_topics = lda_model.get_document_topics(corpus[doc_id], minimum_probability=0)
for topic_id, probability in sorted(doc_topics, key=lambda item: item[1], reverse=True)[:5]:
    print(f"Topic {topic_id}: probability={probability:.3f}")

A história da primeira grande batalha da fase americana da Guerra do Vietnã e os soldados de ambos os lados que a travaram. ...

Topic 13: probability=0.334
Topic 15: probability=0.248
Topic 17: probability=0.222
Topic 1: probability=0.132
Topic 0: probability=0.003


**Solution / interpretation.** These values are the document-topic probabilities and sum to approximately 1 when all topics are included. A document may have meaningful probability across several topics, consistent with LDA’s mixture assumption.

### 8 — Discuss topic membership

What score is high enough to say a document “belongs” to a topic? There is no fixed universal cutoff. A topic may be the document’s strongest theme even when its probability is modest, especially when the document spreads probability across several topics. Consider ranking topics, comparing the top score with the next one, and reading the document before making a claim.

In [29]:
def top_document_topics(model, bow, n=3):
    distribution = model.get_document_topics(bow, minimum_probability=0)
    return sorted(distribution, key=lambda item: item[1], reverse=True)[:n]

for doc_id in [0, 1, 2]:
    print(f"Document {doc_id}:", top_document_topics(lda_model, corpus[doc_id]))

Document 0: [(13, np.float32(0.3341348)), (15, np.float32(0.24703047)), (17, np.float32(0.22150995))]
Document 1: [(18, np.float32(0.4235009)), (14, np.float32(0.40563583)), (3, np.float32(0.13461179))]
Document 2: [(20, np.float32(0.54927707)), (10, np.float32(0.25474432)), (0, np.float32(0.17180052))]


**Solution / discussion.** The highest-probability topic is the model’s strongest topic assignment for that document, but it is not a definitive classification. A practical threshold depends on the use case and should be checked against representative examples. Ranking and inspecting the top few topics is usually more informative than imposing an arbitrary cutoff.

### 9 — Create an interactive pyLDAvis visualization

pyLDAvis presents two complementary views: topic circles summarize prevalence and inter-topic distance, while the term panel shows terms relevant to a selected topic. Circle size reflects topic prevalence; distance between circles approximates topic dissimilarity in the projected view. The visualization is a diagnostic aid, not proof that a topic is valid.

If pyLDAvis is not installed in your environment, install it once with `%pip install pyLDAvis` and restart the kernel if prompted.

In [30]:
%pip install pyLDAvis


[notice] A new release of pip is available: 24.3.1 -> 26.2.2
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [31]:
pip install --upgrade pyLDAvis scikit-learn


[notice] A new release of pip is available: 24.3.1 -> 26.2.2
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [32]:
import warnings

warnings.filterwarnings(
    "ignore",
    category=FutureWarning,
    module="sklearn.manifold._mds"
)

In [33]:
import pyLDAvis
import pyLDAvis.gensim_models as gensimvisualize

pyLDAvis.enable_notebook(local=True)

Prepare the visualization using the trained Gensim model, its bag-of-words corpus, and dictionary. Then display it in the notebook. Review the topics with the top words and representative documents from the previous steps.

In [34]:
lda_visual = gensimvisualize.prepare(
    lda_model,
    corpus,
    id2word,
    mds="mmds",
    sort_topics=False
)
pyLDAvis.display(lda_visual)

**Final discussion.** Do topics that appear close in the map share similar words? Are any topics nearly indistinguishable or represented by only a few generic terms? Use the map to decide which topics deserve closer inspection, then return to their top words and high-scoring documents. Topic quality requires interpretation alongside the visualization.